# Refactored Training Notebook

This notebook keeps the useful pieces from the original file and removes duplicated data preparation/training blocks. The flow is now: configuration -> data loading -> model -> training utilities -> run single or multiple basis functions.


In [ ]:
from pathlib import Path
import math

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as io
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


SEED = 42
DATA_DIR = Path("./data")
GRID_SIZE = 128
NUM_BASIS = 12
TRAIN_SIZE = 3500
TEST_SIZE = 100
BATCH_SIZE = 32
EPOCHS = 60
GRAD_WEIGHT = 0.1
LEARNING_RATE = 1e-3
OUTPUT_DIR = Path("outputs/evaluation/testcode")
MODEL_DIR = Path("best_model")
SAVE_BEST_MODEL = True

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## Data Loading


In [ ]:
def _read_h5_array(path, key):
    with h5py.File(path, "r") as f:
        arr = f[key][()]
    return arr


def _fix_h5_orientation(arr, leading_dim):
    if arr.shape[0] == leading_dim:
        return np.transpose(arr)
    return arr


def load_pair(data_dir, kappa_name, basis_name, grid_size=128, num_basis=12):
    kappa = _read_h5_array(data_dir / kappa_name, "Ks")
    basis = _read_h5_array(data_dir / basis_name, "bfs")

    kappa = _fix_h5_orientation(kappa, leading_dim=1)
    basis = _fix_h5_orientation(basis, leading_dim=num_basis)

    n_samples = kappa.size // (grid_size * grid_size)
    kappa = kappa.reshape(n_samples, 1, grid_size, grid_size).astype(np.float32)
    basis = basis.reshape(n_samples, grid_size * grid_size, num_basis).astype(np.float32)
    return kappa, basis


def load_all_data(data_dir=DATA_DIR, grid_size=GRID_SIZE, num_basis=NUM_BASIS):
    pairs = [
        ("kappa.mat", "basis.mat"),
        ("kappa2.mat", "basis2.mat"),
    ]
    xs, ys = zip(*(load_pair(data_dir, x_name, y_name, grid_size, num_basis) for x_name, y_name in pairs))
    x_all = np.concatenate(xs, axis=0)
    y_all = np.concatenate(ys, axis=0)
    print("x_all:", x_all.shape, "y_all:", y_all.shape)
    return x_all, y_all


In [ ]:
class KappaBasisDataset(Dataset):
    def __init__(self, x, y, logk=True, normalize_x=True, x_mean=None, x_std=None):
        x = x.astype(np.float32)
        y = y.astype(np.float32)

        if logk:
            x = np.log10(np.clip(x, 1e-12, None))

        if normalize_x:
            self.x_mean = float(x.mean()) if x_mean is None else float(x_mean)
            self.x_std = float(x.std() + 1e-12) if x_std is None else float(x_std)
            x = (x - self.x_mean) / self.x_std
        else:
            self.x_mean = 0.0
            self.x_std = 1.0

        if y.ndim == 2 and y.shape[1] == GRID_SIZE * GRID_SIZE:
            y = y.reshape(-1, 1, GRID_SIZE, GRID_SIZE)
        elif y.ndim == 3 and y.shape[1:] == (GRID_SIZE, GRID_SIZE):
            y = y[:, None, :, :]
        elif y.ndim == 3 and y.shape[1:] == (GRID_SIZE * GRID_SIZE, 1):
            y = y.squeeze(-1).reshape(-1, 1, GRID_SIZE, GRID_SIZE)
        elif y.ndim == 4 and y.shape[1:] == (1, GRID_SIZE, GRID_SIZE):
            pass
        else:
            raise ValueError(f"Unexpected y shape: {y.shape}")

        self.x = torch.from_numpy(x).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


def make_loaders(
    x_all,
    y_all,
    basis_index,
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    logk=True,
):
    """Create train/val/test loaders for one basis function.

    The split is deterministic for a given seed. Training uses train_idx, epoch
    monitoring uses val_idx, and final evaluation/spectral analysis uses the
    100-sample test_idx split out from the original validation pool.
    """
    if not 0 <= basis_index < y_all.shape[-1]:
        raise ValueError(f"basis_index must be in [0, {y_all.shape[-1] - 1}], got {basis_index}")
    if train_size + test_size >= x_all.shape[0]:
        raise ValueError(
            f"train_size + test_size must be smaller than sample count; "
            f"got {train_size} + {test_size} >= {x_all.shape[0]}"
        )

    rng = np.random.default_rng(seed)
    indices = rng.permutation(x_all.shape[0])
    train_idx = indices[:train_size]
    eval_idx = indices[train_size:]
    test_idx = eval_idx[:test_size]
    val_idx = eval_idx[test_size:]

    y_basis = y_all[:, :, basis_index]
    train_dataset = KappaBasisDataset(x_all[train_idx], y_basis[train_idx], logk=logk)

    dataset_kwargs = {
        "logk": logk,
        "x_mean": train_dataset.x_mean,
        "x_std": train_dataset.x_std,
    }
    val_dataset = KappaBasisDataset(x_all[val_idx], y_basis[val_idx], **dataset_kwargs)
    test_dataset = KappaBasisDataset(x_all[test_idx], y_basis[test_idx], **dataset_kwargs)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


## Model


In [ ]:
class FASpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2, use_softmax=False):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.use_softmax = use_softmax

        scale = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))
        self.attn1 = nn.Parameter(torch.ones(1, out_channels, modes1, modes2))
        self.attn2 = nn.Parameter(torch.ones(1, out_channels, modes1, modes2))

    @staticmethod
    def compl_mul2d(x, weights):
        return torch.einsum("bixy,ioxy->boxy", x, weights)

    def get_attention(self, attn, m1, m2):
        a = attn[:, :, :m1, :m2]
        if self.use_softmax:
            a = a.reshape(1, self.out_channels, -1)
            a = torch.softmax(a, dim=-1)
            return a.reshape(1, self.out_channels, m1, m2)
        return torch.sigmoid(a)

    def forward(self, x):
        batch_size, _, height, width = x.shape
        x_ft = torch.fft.rfft2(x, norm="ortho")
        out_ft = torch.zeros(
            batch_size,
            self.out_channels,
            height,
            width // 2 + 1,
            dtype=torch.cfloat,
            device=x.device,
        )

        m1 = min(self.modes1, height)
        m2 = min(self.modes2, width // 2 + 1)

        y_pos = self.compl_mul2d(x_ft[:, :, :m1, :m2], self.weights1[:, :, :m1, :m2])
        y_neg = self.compl_mul2d(x_ft[:, :, -m1:, :m2], self.weights2[:, :, :m1, :m2])
        out_ft[:, :, :m1, :m2] = y_pos * self.get_attention(self.attn1, m1, m2)
        out_ft[:, :, -m1:, :m2] = y_neg * self.get_attention(self.attn2, m1, m2)
        return torch.fft.irfft2(out_ft, s=(height, width), norm="ortho")


class FAFNO2d(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, modes1=16, modes2=16, width=32, padding=8, use_softmax=False):
        super().__init__()
        self.padding = padding
        self.fc0 = nn.Conv2d(in_channels + 2, width, kernel_size=1)

        self.convs = nn.ModuleList([
            FASpectralConv2d(width, width, modes1, modes2, use_softmax=use_softmax)
            for _ in range(4)
        ])
        self.ws = nn.ModuleList([nn.Conv2d(width, width, kernel_size=1) for _ in range(4)])
        self.fc1 = nn.Conv2d(width, 128, kernel_size=1)
        self.fc2 = nn.Conv2d(128, out_channels, kernel_size=1)

    @staticmethod
    def get_grid(shape, device):
        batch_size, _, height, width = shape
        gridx = torch.linspace(0, 1, height, device=device).view(1, 1, height, 1).repeat(batch_size, 1, 1, width)
        gridy = torch.linspace(0, 1, width, device=device).view(1, 1, 1, width).repeat(batch_size, 1, height, 1)
        return torch.cat([gridx, gridy], dim=1)

    def forward(self, x):
        grid = self.get_grid(x.shape, x.device)
        x = self.fc0(torch.cat([x, grid], dim=1))

        if self.padding > 0:
            x = F.pad(x, [0, self.padding, 0, self.padding])

        for conv, w in zip(self.convs[:-1], self.ws[:-1]):
            x = F.gelu(conv(x) + w(x))
        x = self.convs[-1](x) + self.ws[-1](x)

        if self.padding > 0:
            x = x[..., :-self.padding, :-self.padding]

        x = F.gelu(self.fc1(x))
        return self.fc2(x)


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))

    def forward(self, x):
        return self.block(x)


class AttentionGate(nn.Module):
    def __init__(self, gate_channels, skip_channels, hidden_channels):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(gate_channels, hidden_channels, kernel_size=1, bias=True),
            nn.BatchNorm2d(hidden_channels),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(skip_channels, hidden_channels, kernel_size=1, bias=True),
            nn.BatchNorm2d(hidden_channels),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(hidden_channels, 1, kernel_size=1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )

    def forward(self, gate, skip):
        psi = F.gelu(self.W_g(gate) + self.W_x(skip))
        return skip * self.psi(psi)


class UpAtt(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            up_channels = in_channels
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            up_channels = in_channels // 2

        self.att = AttentionGate(up_channels, skip_channels, max(out_channels // 2, 1))
        self.conv = DoubleConv(up_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        diff_y = skip.size(2) - x.size(2)
        diff_x = skip.size(3) - x.size(3)
        x = F.pad(x, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])
        skip = self.att(x, skip)
        return self.conv(torch.cat([skip, x], dim=1))


class AttentionUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=32, bilinear=True, use_fno=True):
        super().__init__()
        self.pre = FAFNO2d(in_channels, in_channels, modes1=32, modes2=32, width=24, padding=4) if use_fno else nn.Identity()

        self.inc = DoubleConv(in_channels, base_channels)
        self.down1 = Down(base_channels, base_channels * 2)
        self.down2 = Down(base_channels * 2, base_channels * 4)
        self.down3 = Down(base_channels * 4, base_channels * 8)
        self.down4 = Down(base_channels * 8, base_channels * 16)

        self.up1 = UpAtt(base_channels * 16, base_channels * 8, base_channels * 8, bilinear)
        self.up2 = UpAtt(base_channels * 8, base_channels * 4, base_channels * 4, bilinear)
        self.up3 = UpAtt(base_channels * 4, base_channels * 2, base_channels * 2, bilinear)
        self.up4 = UpAtt(base_channels * 2, base_channels, base_channels, bilinear)
        self.outc = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.pre(x)
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)


## Training Utilities


In [ ]:
def batch_r2_score(pred, target, eps=1e-12):
    pred = pred.view(pred.size(0), -1)
    target = target.view(target.size(0), -1)
    ss_res = ((target - pred) ** 2).sum(dim=1)
    ss_tot = ((target - target.mean(dim=1, keepdim=True)) ** 2).sum(dim=1)
    return (1.0 - ss_res / (ss_tot + eps)).mean().item()


def gradient_loss(pred, target):
    dx_pred = pred[:, :, 1:, :] - pred[:, :, :-1, :]
    dy_pred = pred[:, :, :, 1:] - pred[:, :, :, :-1]
    dx_true = target[:, :, 1:, :] - target[:, :, :-1, :]
    dy_true = target[:, :, :, 1:] - target[:, :, :, :-1]
    return F.mse_loss(dx_pred, dx_true) + F.mse_loss(dy_pred, dy_true)


def run_epoch(model, loader, optimizer=None, device=device, grad_weight=GRAD_WEIGHT):
    is_train = optimizer is not None
    model.train(is_train)
    running = {"loss": 0.0, "mse": 0.0, "r2": 0.0}

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for x, y in tqdm(loader, desc="Train" if is_train else "Val", leave=False):
            x = x.to(device)
            y = y.to(device)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            pred = model(x)
            if pred.shape != y.shape:
                raise ValueError(f"pred shape {pred.shape} != y shape {y.shape}")

            mse = F.mse_loss(pred, y)
            loss = mse + grad_weight * gradient_loss(pred, y)

            if is_train:
                loss.backward()
                optimizer.step()

            running["loss"] += loss.item()
            running["mse"] += mse.item()
            running["r2"] += batch_r2_score(pred.detach(), y.detach())

    num_batches = max(len(loader), 1)
    return {key: value / num_batches for key, value in running.items()}


def fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device=device,
    epochs=EPOCHS,
    grad_weight=GRAD_WEIGHT,
    save_path="best_model.pth",
    save=False,
    mat_save_path="training_history.mat",
):
    best_val_loss = float("inf")
    history = {
        "train_loss": [],
        "train_mse": [],
        "train_r2": [],
        "val_loss": [],
        "val_mse": [],
        "val_r2": [],
    }

    model.to(device)
    save_path = Path(save_path)

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, optimizer, device, grad_weight)
        val_metrics = run_epoch(model, val_loader, None, device, grad_weight)

        for metric_name in ("loss", "mse", "r2"):
            history[f"train_{metric_name}"].append(train_metrics[metric_name])
            history[f"val_{metric_name}"].append(val_metrics[metric_name])

        print(
            f"Epoch [{epoch}/{epochs}] | "
            f"Train Loss: {train_metrics['loss']:.6f}, "
            f"Train MSE: {train_metrics['mse']:.6f}, "
            f"Train R2: {train_metrics['r2']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.6f}, "
            f"Val MSE: {val_metrics['mse']:.6f}, "
            f"Val R2: {val_metrics['r2']:.4f}"
        )

        if save and val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            save_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), save_path)
            print(f"Saved best model to {save_path}")

    io.savemat(str(mat_save_path), history)
    print(f"Training history saved to {mat_save_path}")
    return history


In [ ]:
@torch.no_grad()
def visualize_predictions(model, loader, device=device, num_samples=3):
    model.eval()
    x, y = next(iter(loader))
    x = x.to(device)
    y = y.to(device)
    pred = model(x)

    for i in range(min(num_samples, x.size(0))):
        x_i = x[i, 0].cpu().numpy()
        y_i = y[i, 0].cpu().numpy()
        p_i = pred[i, 0].cpu().numpy()
        e_i = np.abs(p_i - y_i)

        fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
        for ax, image, title in zip(axes, [x_i, y_i, p_i, e_i], ["Input", "True", "Pred", "Abs Error"]):
            im = ax.imshow(image, cmap="jet")
            ax.set_title(title)
            ax.axis("off")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.show()


## Load Data


In [ ]:
x_all, y_all = load_all_data()


## Train One Basis Function


In [ ]:
# basis_index is zero-based: 3 means the 4th basis function.
BASIS_INDEX = 3

train_loader, val_loader, test_loader = make_loaders(
    x_all,
    y_all,
    basis_index=BASIS_INDEX,
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

model = AttentionUNet(1, 1, base_channels=32).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    grad_weight=GRAD_WEIGHT,
    save_path=MODEL_DIR / f"best_fafno_unet_basis_{BASIS_INDEX + 1}.pth",
    save=SAVE_BEST_MODEL,
    mat_save_path=f"history_basis_{BASIS_INDEX + 1}.mat",
)

visualize_predictions(model, test_loader, device, num_samples=3)


## Spectral Analysis

这一部分用于检查模型在频域里的表现，不只是看像素级 MSE。basis 输出原始排列是 coarse element 内部的局部 patch 顺序，所以先用 `as_physical_field_batch` 把它还原成真正的 `128 x 128` 物理场，再做 FFT。

主要看四类性质：

- `2D log amplitude`：展示单个样本的二维频谱，看预测是否保留了主频方向、纹理和高频细节。
- `radial power E(k)`：把二维频谱按半径平均，比较不同波数 `k` 上的能量分布，能看出模型是否偏向过平滑或高频噪声。
- `relative spectral error` 和 `bandwise spectral error`：按频率/频带分解误差，定位低频、中频还是高频更难学。
- `cross-basis consistency`：比较 11 个 basis 的频谱形状一致性，看模型是否对不同 basis 都保持相似的频域规律。


In [ ]:
def _to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def as_field_batch(x):
    """Convert (H,W), (N,H,W), or (N,1,H,W) into (N,H,W)."""
    arr = _to_numpy(x).astype(np.float64, copy=False)
    if arr.ndim == 2:
        return arr[None, :, :]
    if arr.ndim == 3:
        return arr
    if arr.ndim == 4 and arr.shape[1] == 1:
        return arr[:, 0, :, :]
    if arr.ndim == 4 and arr.shape[-1] == 1:
        return arr[..., 0]
    raise ValueError(f"Unexpected field shape: {arr.shape}")


def basis_raw_to_physical_field(field, coarse_shape=(32, 32), local_shape=(4, 4)):
    """Rebuild the physical 128x128 field from coarse-cell ordered basis values."""
    arr = _to_numpy(field).astype(np.float64, copy=False)
    if arr.ndim == 2 and arr.shape == (coarse_shape[0] * local_shape[0], coarse_shape[1] * local_shape[1]):
        arr = arr.reshape(-1)
    if arr.ndim != 1:
        raise ValueError(f"Expected one flattened or 2D basis field, got {arr.shape}")
    expected = coarse_shape[0] * coarse_shape[1] * local_shape[0] * local_shape[1]
    if arr.size != expected:
        raise ValueError(f"Expected {expected} values, got {arr.size}")
    blocks = arr.reshape(coarse_shape[0], coarse_shape[1], local_shape[0], local_shape[1])
    return blocks.transpose(0, 2, 1, 3).reshape(coarse_shape[0] * local_shape[0], coarse_shape[1] * local_shape[1])


def as_physical_field_batch(x, coarse_shape=(32, 32), local_shape=(4, 4)):
    """Convert model/target basis output to physical (N,128,128) fields for spectral analysis."""
    raw = as_field_batch(x)
    return np.stack([
        basis_raw_to_physical_field(field, coarse_shape=coarse_shape, local_shape=local_shape)
        for field in raw
    ], axis=0)


def coarse_element_patch(field, coarse_elem_idx, local_shape=(4, 4), one_based=True):
    """Extract one 4x4 local basis patch from the raw coarse-cell ordered output."""
    arr = as_field_batch(field)[0].reshape(-1)
    idx = coarse_elem_idx - 1 if one_based else coarse_elem_idx
    patch_size = local_shape[0] * local_shape[1]
    start = idx * patch_size
    end = start + patch_size
    if start < 0 or end > arr.size:
        raise IndexError(f"coarse_elem_idx={coarse_elem_idx} is outside 1..{arr.size // patch_size}")
    return arr[start:end].reshape(local_shape)


def gradient_mse_eval(pred, target):
    dx_pred = pred[:, :, 1:, :] - pred[:, :, :-1, :]
    dy_pred = pred[:, :, :, 1:] - pred[:, :, :, :-1]
    dx_true = target[:, :, 1:, :] - target[:, :, :-1, :]
    dy_true = target[:, :, :, 1:] - target[:, :, :, :-1]
    return F.mse_loss(dx_pred, dx_true) + F.mse_loss(dy_pred, dy_true)


@torch.no_grad()
def evaluate_model_on_loader(model, loader, device=device, return_predictions=True):
    """Return metrics and optionally y_true/y_pred arrays with shape (N,H,W)."""
    model.eval()
    n_samples = 0
    running_mse = 0.0
    running_r2 = 0.0
    num_batches = 0
    sum_mae = 0.0
    sum_rel_l2 = 0.0
    sum_grad_mse = 0.0
    max_abs_error = 0.0
    y_true_list, y_pred_list = [], []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        pred = model(x)

        if pred.shape != y.shape:
            raise ValueError(f"pred shape {pred.shape} != y shape {y.shape}")

        bsz = y.size(0)
        mse = F.mse_loss(pred, y)
        pred_flat = pred.reshape(bsz, -1)
        y_flat = y.reshape(bsz, -1)
        sample_mae = (pred_flat - y_flat).abs().mean(dim=1)
        sample_rel_l2 = torch.linalg.norm(pred_flat - y_flat, dim=1) / (torch.linalg.norm(y_flat, dim=1) + 1e-12)

        running_mse += mse.item()
        running_r2 += batch_r2_score(pred, y)
        sum_mae += sample_mae.sum().item()
        sum_rel_l2 += sample_rel_l2.sum().item()
        sum_grad_mse += gradient_mse_eval(pred, y).item() * bsz
        max_abs_error = max(max_abs_error, (pred - y).abs().max().item())
        n_samples += bsz
        num_batches += 1

        if return_predictions:
            y_true_list.append(y.detach().cpu())
            y_pred_list.append(pred.detach().cpu())

    mse_avg = running_mse / max(num_batches, 1)
    metrics = {
        "mse": mse_avg,
        "rmse": math.sqrt(mse_avg),
        "mae": sum_mae / max(n_samples, 1),
        "r2": running_r2 / max(num_batches, 1),
        "relative_l2": sum_rel_l2 / max(n_samples, 1),
        "grad_mse": sum_grad_mse / max(n_samples, 1),
        "max_abs_error": max_abs_error,
        "n_samples": n_samples,
    }

    if not return_predictions:
        return metrics

    y_true = torch.cat(y_true_list, dim=0).numpy().astype(np.float32)
    y_pred = torch.cat(y_pred_list, dim=0).numpy().astype(np.float32)
    return metrics, as_field_batch(y_true).astype(np.float32), as_field_batch(y_pred).astype(np.float32)


In [ ]:
def frequency_radius(shape):
    h, w = shape
    ky = np.fft.fftshift(np.fft.fftfreq(h) * h)
    kx = np.fft.fftshift(np.fft.fftfreq(w) * w)
    grid_y, grid_x = np.meshgrid(ky, kx, indexing="ij")
    return np.sqrt(grid_x ** 2 + grid_y ** 2)


def make_radial_bins(shape, nbins=50, k_max=None, log_bins=False, include_dc=False):
    if k_max is None:
        k_max = min(shape) / 2.0
    if not log_bins:
        start = -0.5 if include_dc else 0.5
        edges = np.linspace(start, k_max + 0.5, nbins + 1)
    elif include_dc:
        edges = np.concatenate([[-0.5, 0.5], np.logspace(np.log10(1.5), np.log10(k_max + 0.5), nbins - 1)])
    else:
        edges = np.logspace(np.log10(0.5), np.log10(k_max + 0.5), nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return edges, centers


def compute_2d_amplitude(field, remove_mean=False):
    """Centered 2D FFT amplitude for one (H,W) field."""
    arr = as_physical_field_batch(field)[0]
    if remove_mean:
        arr = arr - arr.mean()
    fft = np.fft.fftshift(np.fft.fft2(arr, norm="ortho"))
    return np.abs(fft)


def radial_power_spectrum(field, nbins=50, k_max=None, log_bins=False, include_dc=False, remove_mean=False):
    """Radially averaged power spectrum E(k)."""
    arr = as_physical_field_batch(field)[0]
    if remove_mean:
        arr = arr - arr.mean()
    power_2d = np.abs(np.fft.fftshift(np.fft.fft2(arr, norm="ortho"))) ** 2
    radius = frequency_radius(arr.shape)
    edges, centers = make_radial_bins(arr.shape, nbins=nbins, k_max=k_max, log_bins=log_bins, include_dc=include_dc)
    power = np.full(nbins, np.nan)
    for i in range(nbins):
        mask = (radius >= edges[i]) & (radius < edges[i + 1])
        if np.any(mask):
            power[i] = power_2d[mask].mean()
    return centers, power


def batch_radial_power_spectrum(fields, nbins=50, remove_mean=False):
    fields = as_field_batch(fields)
    all_power = []
    k = None
    for field in fields:
        k, power = radial_power_spectrum(field, nbins=nbins, remove_mean=remove_mean)
        all_power.append(power)
    all_power = np.asarray(all_power)
    return {"k": k, "mean": np.nanmean(all_power, axis=0), "std": np.nanstd(all_power, axis=0), "all": all_power}


def spectral_error_decomposition(y_true, y_pred, nbins=50, remove_mean=False, eps=1e-30):
    """Relative Fourier-domain error by radial wavenumber."""
    true_arr = as_physical_field_batch(y_true)[0]
    pred_arr = as_physical_field_batch(y_pred)[0]
    if true_arr.shape != pred_arr.shape:
        raise ValueError(f"Shape mismatch: true {true_arr.shape}, pred {pred_arr.shape}")
    if remove_mean:
        true_arr = true_arr - true_arr.mean()
        pred_arr = pred_arr - pred_arr.mean()

    true_fft = np.fft.fftshift(np.fft.fft2(true_arr, norm="ortho"))
    pred_fft = np.fft.fftshift(np.fft.fft2(pred_arr, norm="ortho"))
    radius = frequency_radius(true_arr.shape)
    edges, centers = make_radial_bins(true_arr.shape, nbins=nbins, log_bins=False)
    rel_error = np.full(nbins, np.nan)

    for i in range(nbins):
        mask = (radius >= edges[i]) & (radius < edges[i + 1])
        if np.any(mask):
            numerator = np.sum(np.abs(pred_fft[mask] - true_fft[mask]) ** 2)
            denominator = np.sum(np.abs(true_fft[mask]) ** 2) + eps
            rel_error[i] = numerator / denominator
    return centers, rel_error


def batch_spectral_error_decomposition(y_true, y_pred, nbins=50, remove_mean=False):
    true_arr = as_field_batch(y_true)
    pred_arr = as_field_batch(y_pred)
    if true_arr.shape != pred_arr.shape:
        raise ValueError(f"Shape mismatch: true {true_arr.shape}, pred {pred_arr.shape}")
    all_error = []
    k = None
    for t, p in zip(true_arr, pred_arr):
        k, error = spectral_error_decomposition(t, p, nbins=nbins, remove_mean=remove_mean)
        all_error.append(error)
    all_error = np.asarray(all_error)
    return {"k": k, "mean": np.nanmean(all_error, axis=0), "std": np.nanstd(all_error, axis=0), "all": all_error}


def bandwise_spectral_error(y_true, y_pred, bands=None, remove_mean=False, eps=1e-30):
    """Bandwise spectral error for one field pair."""
    true_arr = as_physical_field_batch(y_true)[0]
    pred_arr = as_physical_field_batch(y_pred)[0]
    if bands is None:
        bands = [(0, 4), (4, 8), (8, 16), (16, 32), (32, min(true_arr.shape) / 2)]
    if remove_mean:
        true_arr = true_arr - true_arr.mean()
        pred_arr = pred_arr - pred_arr.mean()

    true_fft = np.fft.fftshift(np.fft.fft2(true_arr, norm="ortho"))
    pred_fft = np.fft.fftshift(np.fft.fft2(pred_arr, norm="ortho"))
    radius = frequency_radius(true_arr.shape)
    total_error_energy = np.sum(np.abs(pred_fft - true_fft) ** 2) + eps
    total_true_energy = np.sum(np.abs(true_fft) ** 2) + eps
    rows = []

    for k_low, k_high in bands:
        mask = radius >= k_low if k_high is None else ((radius >= k_low) & (radius < k_high))
        if not np.any(mask):
            continue
        true_energy = np.sum(np.abs(true_fft[mask]) ** 2)
        pred_energy = np.sum(np.abs(pred_fft[mask]) ** 2)
        error_energy = np.sum(np.abs(pred_fft[mask] - true_fft[mask]) ** 2)
        rows.append({
            "band": f"[{k_low:g}, {'inf' if k_high is None else f'{k_high:g}'})",
            "k_low": k_low,
            "k_high": np.inf if k_high is None else k_high,
            "true_energy": true_energy,
            "pred_energy": pred_energy,
            "error_energy": error_energy,
            "rel_error": error_energy / (true_energy + eps),
            "pred_true_energy_ratio": pred_energy / (true_energy + eps),
            "error_fraction": error_energy / total_error_energy,
            "true_energy_fraction": true_energy / total_true_energy,
        })
    return pd.DataFrame(rows)


def batch_bandwise_spectral_error(y_true, y_pred, basis_id=None, model_name="testcode", bands=None, remove_mean=False):
    true_arr = as_field_batch(y_true)
    pred_arr = as_field_batch(y_pred)
    rows = []
    for sample_idx, (t, p) in enumerate(zip(true_arr, pred_arr)):
        df = bandwise_spectral_error(t, p, bands=bands, remove_mean=remove_mean)
        df.insert(0, "sample_idx", sample_idx)
        rows.append(df)
    long_df = pd.concat(rows, ignore_index=True)
    value_cols = ["true_energy", "pred_energy", "error_energy", "rel_error", "pred_true_energy_ratio", "error_fraction", "true_energy_fraction"]
    agg = long_df.groupby(["band", "k_low", "k_high"], as_index=False)[value_cols].agg(["mean", "std"])
    agg.columns = ["_".join([c for c in col if c]) if isinstance(col, tuple) else col for col in agg.columns.to_flat_index()]
    agg.insert(0, "model", model_name)
    if basis_id is not None:
        agg.insert(0, "basis_id", basis_id)
    return agg


def aggregate_spectral_results_by_basis(y_true_by_basis, y_pred_by_basis, nbins=50, bands=None, output_dir=OUTPUT_DIR, model_name="testcode", remove_mean=False):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    radial_power = {}
    spectral_error = {}
    bandwise_rows = []

    for basis_id in sorted(y_true_by_basis):
        y_true = y_true_by_basis[basis_id]
        y_pred = y_pred_by_basis[basis_id]
        radial_power[basis_id] = {
            "true": batch_radial_power_spectrum(y_true, nbins=nbins, remove_mean=remove_mean),
            "pred": batch_radial_power_spectrum(y_pred, nbins=nbins, remove_mean=remove_mean),
        }
        spectral_error[basis_id] = batch_spectral_error_decomposition(y_true, y_pred, nbins=nbins, remove_mean=remove_mean)
        bandwise_rows.append(batch_bandwise_spectral_error(y_true, y_pred, basis_id=basis_id, model_name=model_name, bands=bands, remove_mean=remove_mean))

    bandwise_df = pd.concat(bandwise_rows, ignore_index=True)
    bandwise_df.to_csv(output_dir / "bandwise_spectral_error.csv", index=False)
    return {"radial_power": radial_power, "spectral_error": spectral_error, "bandwise": bandwise_df}


def cross_basis_spectral_consistency(y_true_by_basis, y_pred_by_basis, nbins=50, remove_mean=False, eps=1e-30):
    """Compare the radial spectral shape of prediction vs truth across all basis functions."""
    rows = []
    for basis_id in sorted(y_true_by_basis):
        true_spec = batch_radial_power_spectrum(y_true_by_basis[basis_id], nbins=nbins, remove_mean=remove_mean)
        pred_spec = batch_radial_power_spectrum(y_pred_by_basis[basis_id], nbins=nbins, remove_mean=remove_mean)
        for k, true_power, pred_power in zip(true_spec["k"], true_spec["mean"], pred_spec["mean"]):
            true_log = np.log10(max(true_power, eps))
            pred_log = np.log10(max(pred_power, eps))
            rows.append({"basis_id": basis_id, "k": k, "true_log_power": true_log, "pred_log_power": pred_log, "log_power_error": pred_log - true_log})
    spectrum_df = pd.DataFrame(rows)

    per_basis_rows = []
    for basis_id, group in spectrum_df.groupby("basis_id"):
        corr = np.corrcoef(group["true_log_power"], group["pred_log_power"])[0, 1]
        mae_log = np.mean(np.abs(group["log_power_error"]))
        per_basis_rows.append({"basis_id": basis_id, "corr_across_k": corr, "mae_log_power": mae_log})

    per_k_rows = []
    for k, group in spectrum_df.groupby("k"):
        corr = np.corrcoef(group["true_log_power"], group["pred_log_power"])[0, 1]
        mae_log = np.mean(np.abs(group["log_power_error"]))
        per_k_rows.append({"k": k, "corr_across_basis": corr, "mae_log_power": mae_log})

    return {"basis_spectrum": spectrum_df, "per_basis": pd.DataFrame(per_basis_rows), "per_k": pd.DataFrame(per_k_rows)}


In [ ]:
def plot_2d_spectrum_comparison(y_true, preds_dict, save_path=None, remove_mean=False, eps=1e-12):
    true_field = as_physical_field_batch(y_true)[0]
    pred_fields = {name: as_physical_field_batch(pred)[0] for name, pred in preds_dict.items()}
    ncols = 1 + len(pred_fields)
    fig, axes = plt.subplots(2, ncols, figsize=(4 * ncols, 7))
    if ncols == 1:
        axes = axes.reshape(2, 1)

    spatial_vmin = min([true_field.min()] + [p.min() for p in pred_fields.values()])
    spatial_vmax = max([true_field.max()] + [p.max() for p in pred_fields.values()])
    true_for_fft = true_field - true_field.mean() if remove_mean else true_field
    amp_true = np.abs(np.fft.fftshift(np.fft.fft2(true_for_fft, norm="ortho")))
    amp_vmax = np.log10(amp_true + eps).max()

    axes[0, 0].imshow(true_field, cmap="viridis", vmin=spatial_vmin, vmax=spatial_vmax)
    axes[0, 0].set_title("Ground truth")
    axes[0, 0].axis("off")
    axes[1, 0].imshow(np.log10(amp_true + eps), cmap="inferno", vmax=amp_vmax)
    axes[1, 0].set_title("GT log amplitude")
    axes[1, 0].axis("off")

    for col, (name, pred) in enumerate(pred_fields.items(), start=1):
        axes[0, col].imshow(pred, cmap="viridis", vmin=spatial_vmin, vmax=spatial_vmax)
        axes[0, col].set_title(name)
        axes[0, col].axis("off")
        pred_for_fft = pred - pred.mean() if remove_mean else pred
        amp = np.abs(np.fft.fftshift(np.fft.fft2(pred_for_fft, norm="ortho")))
        axes[1, col].imshow(np.log10(amp + eps), cmap="inferno", vmax=amp_vmax)
        axes[1, col].set_title(f"{name} log amplitude")
        axes[1, col].axis("off")

    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_aggregate_spectral_curves(spectral_results, basis_id, save_path=None):
    radial = spectral_results["radial_power"][basis_id]
    error = spectral_results["spectral_error"][basis_id]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for label, color in [("true", "black"), ("pred", "tab:blue")]:
        data = radial[label]
        axes[0].loglog(data["k"], data["mean"], color=color, linewidth=2, label=label)
        axes[0].fill_between(data["k"], np.maximum(data["mean"] - data["std"], 1e-30), data["mean"] + data["std"], color=color, alpha=0.15)
    axes[0].set_xlabel("Wavenumber k")
    axes[0].set_ylabel("Power E(k)")
    axes[0].set_title(f"Basis {basis_id}: radial power")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].loglog(error["k"], error["mean"], color="tab:red", linewidth=2, label="relative error")
    axes[1].fill_between(error["k"], np.maximum(error["mean"] - error["std"], 1e-30), error["mean"] + error["std"], color="tab:red", alpha=0.15)
    axes[1].set_xlabel("Wavenumber k")
    axes[1].set_ylabel("Relative spectral error")
    axes[1].set_title(f"Basis {basis_id}: spectral error")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_bandwise_error(bandwise_df, value_col="rel_error_mean", save_path=None):
    fig, ax = plt.subplots(figsize=(9, 5))
    for basis_id, group in bandwise_df.groupby("basis_id"):
        ax.plot(group["band"], group[value_col], marker="o", linewidth=1.2, alpha=0.8, label=f"B{basis_id}")
    ax.set_yscale("log")
    ax.set_xlabel("Frequency band")
    ax.set_ylabel(value_col)
    ax.set_title("Bandwise spectral error")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(ncol=4, fontsize=8)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_coarse_element_basis_prediction(y_true_by_basis, y_pred_by_basis, sample_idx=0, coarse_elem_idx=1,
                                         group1=range(2, 7), group2=range(7, 13), cmap="Reds"):
    """Plot predicted/reference 4x4 local basis patches on one coarse element."""
    groups = [list(group1), list(group2)]
    max_cols = max(len(g) for g in groups)
    fig, axes = plt.subplots(4, max_cols, figsize=(2.4 * max_cols, 8.5))

    row_specs = [
        (0, "Pred", groups[0], y_pred_by_basis),
        (1, "Ref", groups[0], y_true_by_basis),
        (2, "Pred", groups[1], y_pred_by_basis),
        (3, "Ref", groups[1], y_true_by_basis),
    ]

    for row, row_label, basis_group, data_dict in row_specs:
        for col in range(max_cols):
            ax = axes[row, col]
            if col >= len(basis_group):
                ax.axis("off")
                continue
            basis_id = basis_group[col]
            true_patch = coarse_element_patch(y_true_by_basis[basis_id][sample_idx], coarse_elem_idx)
            pred_patch = coarse_element_patch(y_pred_by_basis[basis_id][sample_idx], coarse_elem_idx)
            patch = pred_patch if row_label == "Pred" else true_patch
            im = ax.contourf(patch, cmap=cmap)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(f"{row_label} B{basis_id}", fontsize=10)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

    fig.suptitle(f"Sample {sample_idx}, coarse element {coarse_elem_idx}: local 4x4 basis patches", y=1.01)
    plt.tight_layout()
    plt.show()


def plot_cross_basis_consistency(consistency, save_path=None):
    per_basis = consistency["per_basis"]
    per_k = consistency["per_k"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].bar(per_basis["basis_id"].astype(str), per_basis["corr_across_k"])
    axes[0].set_ylim(-1, 1)
    axes[0].set_xlabel("Basis id")
    axes[0].set_ylabel("Correlation across k")
    axes[0].set_title("Per-basis spectral-shape consistency")
    axes[0].grid(True, axis="y", alpha=0.3)

    axes[1].semilogx(per_k["k"], per_k["corr_across_basis"], marker="o", linewidth=1.5)
    axes[1].set_ylim(-1, 1)
    axes[1].set_xlabel("Wavenumber k")
    axes[1].set_ylabel("Correlation across bases")
    axes[1].set_title("Cross-basis consistency by frequency")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


## Train 11 Basis Models


In [ ]:
# Train 11 independent models, matching the original range(1, 12) logic.
# basis_index is zero-based, so this trains basis 2 through basis 12.
# Validation monitors training; the 100-sample test split is used for final metrics and spectral analysis below.
BASIS_INDICES = range(1, NUM_BASIS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Best checkpoints will be saved in: {MODEL_DIR.resolve()}")

all_histories = {}
test_metrics_rows = []
y_true_by_basis = {}
y_pred_by_basis = {}

for basis_index in BASIS_INDICES:
    basis_number = basis_index + 1
    print(f"Training basis {basis_number}/{NUM_BASIS}")

    train_loader, val_loader, test_loader = make_loaders(
        x_all,
        y_all,
        basis_index=basis_index,
        train_size=TRAIN_SIZE,
        test_size=TEST_SIZE,
        batch_size=BATCH_SIZE,
        seed=SEED + basis_index,
    )

    model = AttentionUNet(1, 1, base_channels=32).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        epochs=EPOCHS,
        grad_weight=GRAD_WEIGHT,
        save_path=MODEL_DIR / f"best_fafno_unet_basis_{basis_number}.pth",
        save=SAVE_BEST_MODEL,
        mat_save_path=f"history_basis_{basis_number}.mat",
    )

    metrics, y_true, y_pred = evaluate_model_on_loader(model, test_loader, device=device, return_predictions=True)
    test_metrics_rows.append({"basis_id": basis_number, **metrics})
    y_true_by_basis[basis_number] = y_true
    y_pred_by_basis[basis_number] = y_pred
    np.savez_compressed(OUTPUT_DIR / f"predictions_basis{basis_number}.npz", y_true=y_true, y_pred=y_pred)

    all_histories[basis_number] = history
    print(
        f"Finished basis {basis_number}/{NUM_BASIS} | "
        f"Test MSE={metrics['mse']:.6e}, R2={metrics['r2']:.6f}, RelL2={metrics['relative_l2']:.6e}"
    )

test_metrics_df = pd.DataFrame(test_metrics_rows)
test_metrics_df.to_csv(OUTPUT_DIR / "test_metrics_by_basis.csv", index=False)
display(test_metrics_df)


## Run Spectral Analysis

运行完上面的 11 个模型训练 cell 后，`y_true_by_basis` 和 `y_pred_by_basis` 中保存的是每个 basis 在 100 个 test 样本上的真实值和预测值。这里会把这些 test 结果汇总成频谱指标，并保存为 CSV，方便后续画图或写论文表格。


In [ ]:
# Run spectral analysis on the 100-sample test split collected from the 11-model training cell.
NBINS = 50
REMOVE_MEAN_FOR_SPECTRUM = False

spectral_results = aggregate_spectral_results_by_basis(
    y_true_by_basis,
    y_pred_by_basis,
    nbins=NBINS,
    output_dir=OUTPUT_DIR,
    model_name="testcode_fafno_unet",
    remove_mean=REMOVE_MEAN_FOR_SPECTRUM,
)

cross_basis_results = cross_basis_spectral_consistency(
    y_true_by_basis,
    y_pred_by_basis,
    nbins=NBINS,
    remove_mean=REMOVE_MEAN_FOR_SPECTRUM,
)

for name, df in cross_basis_results.items():
    df.to_csv(OUTPUT_DIR / f"cross_basis_{name}.csv", index=False)

display(spectral_results["bandwise"].head())
display(cross_basis_results["per_basis"])


## Spectral Plots

这些图用于从不同角度解释频域误差：`plot_2d_spectrum_comparison` 看单个样本的空间场和频谱图；`plot_aggregate_spectral_curves` 看某个 basis 的平均能量谱和误差谱；`plot_bandwise_error` 横向比较 11 个 basis 在不同频带的误差；`plot_cross_basis_consistency` 检查不同 basis 的频谱形状是否稳定。


In [ ]:
# Example plots. Change basis_id and sample_idx as needed.
basis_id = 2
sample_idx = 0

plot_2d_spectrum_comparison(
    y_true=y_true_by_basis[basis_id][sample_idx],
    preds_dict={"FAFNO-UNet": y_pred_by_basis[basis_id][sample_idx]},
    save_path=OUTPUT_DIR / f"spectrum2d_basis{basis_id}_sample{sample_idx}.png",
    remove_mean=REMOVE_MEAN_FOR_SPECTRUM,
)

plot_coarse_element_basis_prediction(
    y_true_by_basis=y_true_by_basis,
    y_pred_by_basis=y_pred_by_basis,
    sample_idx=sample_idx,
    coarse_elem_idx=np.random.randint(1, 1025),
)

plot_aggregate_spectral_curves(
    spectral_results,
    basis_id=basis_id,
    save_path=OUTPUT_DIR / f"aggregate_spectral_basis{basis_id}.png",
)

plot_bandwise_error(
    spectral_results["bandwise"],
    save_path=OUTPUT_DIR / "bandwise_spectral_error.png",
)

plot_cross_basis_consistency(
    cross_basis_results,
    save_path=OUTPUT_DIR / "cross_basis_spectral_consistency.png",
)
